# LeetCode 3999 — Minimum Number of String Groups Through Transformations bằng Python

- Link: https://leetcode.com/problems/minimum-number-of-string-groups-through-transformations/
- Độ khó: **Hard**
- Chủ đề: Chuỗi, Dạng chuẩn tắc (canonical form), Xoay vòng nhỏ nhất (minimal rotation), Hash Table

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumGroups(self, words: List[str]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Với mỗi chuỗi `s`, tách thành hai dãy: `E` gồm các ký tự ở chỉ số **chẵn** (0, 2, 4, ...) và `O` gồm các ký tự ở chỉ số **lẻ**. Một phép biến đổi **xoay vòng** `E` và `O` **độc lập** (mỗi dãy xoay sang phải bao nhiêu vị trí tuỳ ý) rồi ráp lại. Hai chuỗi là **tương đương** nếu chuỗi này biến thành chuỗi kia qua một phép biến đổi. Chia `words` thành ít nhóm nhất sao cho mọi cặp trong cùng nhóm đều tương đương.

Ý chính: "tương đương" là một **quan hệ tương đương** (phản xạ, đối xứng, bắc cầu), nên số nhóm tối thiểu **đúng bằng số lớp tương đương**. Mỗi lớp được đặc trưng bởi **cặp dạng chuẩn** `(chuẩn(E), chuẩn(O))`, trong đó "chuẩn" của một chuỗi là **xoay vòng nhỏ nhất theo thứ tự từ điển**. Tính xoay vòng nhỏ nhất trong `O(L)` bằng thuật toán hai con trỏ.

Sau notebook này bạn sẽ:

1. Biết cách tách chuỗi chẵn/lẻ bằng slicing `s[::2]` và `s[1::2]`, cùng phép xoay vòng.
2. Chứng minh "tương đương" là quan hệ tương đương, nên đếm lớp = đếm khoá chuẩn khác nhau.
3. Viết vét cạn theo đúng định nghĩa (đồ thị tương đương + chia nhóm tối thiểu) để làm trọng tài.
4. Viết `O(L²)` min-rotation, rồi `O(L)` bằng thuật toán hai con trỏ (đáp án nộp), cùng một cách "chia thùng theo đa tập rồi kiểm tra bằng nhân đôi chuỗi".
5. Nhận ra **bẫy**: chỉ so sánh đa tập (sắp xếp) hoặc xoay cả chuỗi là sai.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Với chuỗi `s`: `E` = ký tự ở chỉ số chẵn (theo thứ tự), `O` = ký tự ở chỉ số lẻ.
- Một **phép biến đổi**: chọn `a`, `b ≥ 0`, xoay phải `E` đi `a` vị trí, xoay phải `O` đi `b` vị trí, rồi đặt lại `E` vào các chỉ số chẵn, `O` vào các chỉ số lẻ.
- Hai chuỗi **tương đương** nếu có một phép biến đổi biến chuỗi này thành chuỗi kia.
- Chia `words` thành số nhóm **ít nhất** sao cho mỗi chuỗi thuộc đúng một nhóm và mọi cặp trong cùng nhóm tương đương. Trả về số nhóm đó.

**Ví dụ 1:** `["ntgwz","zwntg"]` → `1`. `ntgwz`: `E = "ngz"`, `O = "tw"`. Xoay `E` phải 1 vị trí: `"zng"`; xoay `O` phải 1: `"wt"`. Ráp: `z w n t g` = `"zwntg"`.

**Ví dụ 2:** `["abc","cab","bac","acb","bca","cba"]` → `3`: nhóm `{abc, cba}`, `{cab, bac}`, `{acb, bca}`.

**Ví dụ 3:** `["leet","abb","bab","deed","edde","code","bba"]` → `5`: `{abb, bba}`, `{deed, edde}`, `{leet}`, `{bab}`, `{code}`.

**Ràng buộc**

- `1 <= words.length <= 10^5`
- `1 <= words[i].length <= 5·10^5`
- Tổng độ dài các chuỗi `<= 5·10^5`
- Chỉ gồm chữ thường.

Lưu ý: tổng độ dài chỉ `5·10^5`, nên một lời giải **tuyến tính theo tổng độ dài** là đủ; còn `O(L²)` cho một chuỗi dài `5·10^5` là không thể.

In [1]:
from typing import List

EXAMPLES = [
    (["ntgwz", "zwntg"], 1),
    (["abc", "cab", "bac", "acb", "bca", "cba"], 3),
    (["leet", "abb", "bab", "deed", "edde", "code", "bba"], 5),
]
for words, expected in EXAMPLES:
    print(f"words={words} -> kỳ vọng {expected}")

words=['ntgwz', 'zwntg'] -> kỳ vọng 1
words=['abc', 'cab', 'bac', 'acb', 'bca', 'cba'] -> kỳ vọng 3
words=['leet', 'abb', 'bab', 'deed', 'edde', 'code', 'bba'] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Slicing có bước: `s[::2]` và `s[1::2]`

Cú pháp `s[start:stop:step]`. Bỏ trống `start` nghĩa là từ đầu. `s[::2]` lấy các chỉ số `0, 2, 4, ...`; `s[1::2]` lấy `1, 3, 5, ...`.

In [2]:
s = "abcde"
print(s[::2])     # ace  (chỉ số chẵn)
print(s[1::2])    # bd   (chỉ số lẻ)
print("a"[1::2] == "")   # chuỗi dài 1: dãy lẻ rỗng

ace
bd
True


### 3.2. Xoay vòng chuỗi (cyclic shift)

Xoay **phải** `a` vị trí: ký tự ở cuối đưa lên đầu. Dùng slicing: `x[-a:] + x[:-a]` (với `a = 0` cần xử lý riêng vì `x[-0:] = x`). Cách gọn hơn là `a %= len(x)` rồi `x[len(x)-a:] + x[:len(x)-a]`. Xoay trái cũng là một phép xoay phải nên tập tất cả các phép xoay là như nhau.

In [3]:
def rotate_right(x, a):
    if not x:
        return x
    a %= len(x)
    return x[len(x) - a:] + x[:len(x) - a]

x = "abcde"
print([rotate_right(x, a) for a in range(6)])     # a = 5 quay về x
print(rotate_right("ngz", 1), rotate_right("tw", 1))    # zng  wt
print(repr(rotate_right("", 3)))

['abcde', 'eabcd', 'deabc', 'cdeab', 'bcdea', 'abcde']
zng wt
''


### 3.3. `set`/`dict` để đếm khoá khác nhau, và `in` trên chuỗi nhân đôi

- `len(set(keys))` = số khoá phân biệt (khoá phải **băm được**: chuỗi và tuple chuỗi đều được).
- **Mẹo nhân đôi:** `y` là một phép xoay của `x` ⇔ `len(x) == len(y)` và `y in x + x`.

In [4]:
keys = [("ac", "b"), ("ac", "b"), ("bc", "a")]
print(len(set(keys)))                       # 2

x, y = "abcde", "cdeab"
print(len(x) == len(y) and y in x + x)       # True: y là phép xoay của x
print("acbde" in x + x)                      # False

2
True
False


## 4. Ý tưởng nền tảng / trực giác

Phép biến đổi tách chuỗi làm **hai thế giới độc lập**: các ký tự ở chỉ số chẵn chỉ đi quanh trong chỉ số chẵn (xoay vòng), tương tự với lẻ. Do đó `s ~ t` ⇔ `E_t` là một phép xoay của `E_s` **và** `O_t` là một phép xoay của `O_s`.

Tính tay ví dụ 1: `ntgwz` có `E = "ngz"`, `O = "tw"`; `zwntg` có `E = "zng"`, `O = "wt"`. `"zng"` là phép xoay của `"ngz"`; `"wt"` là phép xoay của `"tw"` ⇒ tương đương.

Ví dụ 2: `abc`: `E = "ac"`, `O = "b"`. `cba`: `E = "ca"`, `O = "b"`. `"ca"` là phép xoay của `"ac"` ⇒ cùng nhóm. `cab`: `E = "cb"`, `O = "a"` — `E` khác kiểu `{a,c}` nên khác nhóm.

Hãy viết các hàm tách và kiểm tra tương đương để thử trực tiếp.

In [5]:
def split_eo(s):
    return s[::2], s[1::2]

def is_rotation(x, y):
    return len(x) == len(y) and (y in x + x)

def equivalent_fast(s, t):
    es, os_ = split_eo(s)
    et, ot = split_eo(t)
    return is_rotation(es, et) and is_rotation(os_, ot)

print(split_eo("ntgwz"), split_eo("zwntg"))
print(equivalent_fast("ntgwz", "zwntg"))      # True
print(equivalent_fast("abc", "cba"), equivalent_fast("abc", "cab"))    # True False

('ngz', 'tw') ('zng', 'wt')
True
True False


## 5. Cách 1 — Brute Force theo định nghĩa

**Ý tưởng.** Dựng đúng như đề:

1. `transform(s, a, b)`: xoay `E` phải `a`, xoay `O` phải `b`, ráp lại.
2. `equivalent(s, t)`: tồn tại `(a, b)` sao cho `transform(s, a, b) == t`.
3. Dựng **đồ thị** các cặp tương đương, rồi tìm cách **chia nhóm ít nhất** sao cho mỗi nhóm là một **clique** (mọi cặp tương đương) — bằng đệ quy thử xếp từng chuỗi vào nhóm có sẵn hoặc mở nhóm mới (với danh sách ngắn).

Cách này không dựa vào bất kỳ nhận xét nào về lớp tương đương, nên tin cậy làm trọng tài.

In [6]:
def interleave(E, O):
    n = len(E) + len(O)
    res = [""] * n
    res[0::2] = E
    res[1::2] = O
    return "".join(res)

def transform(s, a, b):
    E, O = split_eo(s)
    return interleave(rotate_right(E, a), rotate_right(O, b))

def equivalent(s, t):
    if len(s) != len(t):
        return False
    E, O = split_eo(s)
    for a in range(max(1, len(E))):
        for b in range(max(1, len(O))):
            if transform(s, a, b) == t:
                return True
    return False

def brute_groups(words):
    n = len(words)
    eq = [[equivalent(words[i], words[j]) for j in range(n)] for i in range(n)]
    best = [n]
    groups = []
    def dfs(i):
        if len(groups) >= best[0]:
            return
        if i == n:
            best[0] = len(groups)
            return
        for g in groups:
            if all(eq[i][j] for j in g):       # i tương đương với mọi phần tử trong nhóm
                g.append(i)
                dfs(i + 1)
                g.pop()
        groups.append([i])
        dfs(i + 1)
        groups.pop()
    dfs(0)
    return best[0]

for words, expected in EXAMPLES:
    print(words, "->", brute_groups(words), "(kỳ vọng", expected, ")")

['ntgwz', 'zwntg'] -> 1 (kỳ vọng 1 )
['abc', 'cab', 'bac', 'acb', 'bca', 'cba'] -> 3 (kỳ vọng 3 )
['leet', 'abb', 'bab', 'deed', 'edde', 'code', 'bba'] -> 5 (kỳ vọng 5 )


**Độ phức tạp.** Kiểm tra một cặp: `O(|E|·|O|·L)`; chia nhóm tối thiểu là bài toán phủ clique, cỡ `O(Bell(n))` — chỉ dùng cho `n ≤ 7`, chuỗi ngắn. **Nút thắt:** ta kiểm tra từng cặp và thử mọi cách chia, trong khi "tương đương" là một **quan hệ tương đương** nên chỉ cần gán mỗi chuỗi một **nhãn lớp**.

## 6. Bước đột phá — quan hệ tương đương và dạng chuẩn tắc

**Bổ đề (quan hệ tương đương).** "`s ~ t`" thoả:

- *Phản xạ*: dùng `a = b = 0`.
- *Đối xứng*: nếu xoay phải `a` biến `E_s` thành `E_t` thì xoay phải `|E| − a` biến ngược lại.
- *Bắc cầu*: hợp hai phép xoay là một phép xoay (`a₁ + a₂`).

Do đó các chuỗi chia thành các **lớp** và "một nhóm mà mọi cặp tương đương" chính là **nằm trong một lớp**. Số nhóm tối thiểu = số lớp (mỗi lớp là một nhóm hợp lệ, và không thể gộp hai lớp khác nhau).

**Dạng chuẩn tắc.** Hai dãy ký tự là phép xoay của nhau ⇔ chúng có cùng **xoay vòng nhỏ nhất theo từ điển** `lyndon(x) = min(x[i:] + x[:i])`. Vậy khoá lớp của `s` là cặp `(lyndon(E), lyndon(O))`.

Số nhóm = số khoá **khác nhau**. Ta kiểm chứng điều này với brute force trước khi tối ưu cách tính `lyndon`.

In [7]:
def min_rotation_naive(x):
    if not x:
        return x
    return min(x[i:] + x[:i] for i in range(len(x)))

def key_naive(s):
    E, O = split_eo(s)
    return (min_rotation_naive(E), min_rotation_naive(O))

print(min_rotation_naive("cabca"), min_rotation_naive("ngz"), min_rotation_naive("zng"))
print(key_naive("ntgwz"), key_naive("zwntg"))

abcac gzn gzn
('gzn', 'tw') ('gzn', 'tw')


## 7. Các cách tiếp theo

- **Cách 2:** khoá `(min_rotation_naive(E), min_rotation_naive(O))`, `O(L²)` mỗi chuỗi.
- **Cách 3:** tính xoay vòng nhỏ nhất bằng **thuật toán hai con trỏ** `O(L)` — đáp án nộp.
- **Cách 4:** chia **thùng** theo đa tập (đếm ký tự) rồi trong mỗi thùng gom theo "`y in x + x`" — minh hoạ vì sao đa tập **chưa đủ**.

### 7.1. Cách 2 — Khoá bằng min-rotation ngây thơ

**Giải thích.** Với mỗi chuỗi, tách `E, O`, tính xoay vòng nhỏ nhất của mỗi dãy bằng cách thử mọi vị trí bắt đầu `i` (`x[i:] + x[:i]`) rồi lấy `min`. Đếm số khoá khác nhau bằng `set`.

In [8]:
def groups_naive(words):
    return len({key_naive(w) for w in words})

for words, expected in EXAMPLES:
    assert groups_naive(words) == expected
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp.** Với một dãy dài `ℓ`: `ℓ` phép xoay, mỗi phép `O(ℓ)` để tạo chuỗi và so sánh → `O(ℓ²)`. Tổng `O(Σ L²)`: với một chuỗi `5·10^5` ký tự là cỡ `10^11` — quá chậm. Ta cần `O(ℓ)`.

### 7.2. Cách 3 — Xoay vòng nhỏ nhất bằng hai con trỏ `O(ℓ)` (đáp án nộp)

**Ý tưởng (thuật toán "biểu diễn nhỏ nhất").** Xét hai ứng viên điểm bắt đầu `i` và `j` (`i ≠ j`), và đếm `k` = số ký tự khớp liên tiếp khi so sánh xoay từ `i` và xoay từ `j`:

- Nếu `x[(i+k) % n] == x[(j+k) % n]`: tăng `k`.
- Nếu `x[(i+k) % n] > x[(j+k) % n]`: mọi điểm bắt đầu `i, i+1, ..., i+k` đều **không thể** là xoay nhỏ nhất (vì với mỗi `i+t`, có `j+t` cho xoay nhỏ hơn) → `i += k + 1`.
- Ngược lại: `j += k + 1`.
- Nếu `i == j` thì `j += 1`. Đặt lại `k = 0`.

Dừng khi `i`, `j` hoặc `k` chạm `n`. Điểm bắt đầu nhỏ nhất là `min(i, j)`. Mỗi bước hoặc tăng `k` hoặc đẩy `i`/`j` tiến thêm `k+1`, nên tổng bước `O(n)`.

In [9]:
def min_rotation_start(x):
    n = len(x)
    i, j, k = 0, 1, 0
    while i < n and j < n and k < n:
        a, b = x[(i + k) % n], x[(j + k) % n]
        if a == b:
            k += 1
            continue
        if a > b:
            i += k + 1
        else:
            j += k + 1
        if i == j:
            j += 1
        k = 0
    return min(i, j)

def min_rotation_fast(x):
    if len(x) <= 1:
        return x
    p = min_rotation_start(x)
    return x[p:] + x[:p]

for x in ["cabca", "ngz", "zng", "aaaa", "abab", "baab", "a", "", "bcab"]:
    assert min_rotation_fast(x) == min_rotation_naive(x), x
print("min_rotation_fast khớp ngây thơ trên các mẫu cố định")

min_rotation_fast khớp ngây thơ trên các mẫu cố định


In [10]:
def groups_fast(words):
    keys = set()
    for w in words:
        E, O = w[::2], w[1::2]
        keys.add((min_rotation_fast(E), min_rotation_fast(O)))
    return len(keys)

for words, expected in EXAMPLES:
    assert groups_fast(words) == expected
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


**Độ phức tạp.** `O(L)` cho mỗi chuỗi độ dài `L` (tổng `O(Σ L)`), `O(L)` bộ nhớ cho khoá (tổng `O(Σ L)`).

### 7.3. Cách 4 — Chia thùng theo đa tập, rồi kiểm tra bằng nhân đôi

**Ý tưởng.** Xoay vòng giữ nguyên **đa tập ký tự**, nên hai chuỗi cùng lớp phải có cùng `(sorted(E), sorted(O))`. Ta dùng nó làm **bộ lọc thô** để chia thùng; trong mỗi thùng, gom từng chuỗi vào nhóm đầu tiên mà đại diện của nó tương đương (kiểm tra `y in x + x`). Cách này **đúng** nhưng chậm hơn (kiểm tra chuỗi con, tạo `x + x`) và có thể `O(k²)` trong một thùng lớn. Nó cho thấy: đa tập chỉ là điều kiện **cần**, không đủ.

In [11]:
def groups_bucket(words):
    buckets = {}
    for w in words:
        E, O = w[::2], w[1::2]
        sig = ("".join(sorted(E)), "".join(sorted(O)))
        buckets.setdefault(sig, []).append((E, O))
    total = 0
    for items in buckets.values():
        reps = []                                   # đại diện của các nhóm trong thùng
        for E, O in items:
            for rE, rO in reps:
                if is_rotation(rE, E) and is_rotation(rO, O):
                    break
            else:
                reps.append((E, O))
        total += len(reps)
    return total

for words, expected in EXAMPLES:
    assert groups_bucket(words) == expected
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


**Độ phức tạp.** Sắp xếp ký tự: `O(L log L)`; so sánh trong thùng: tới `O(k · g · L)` với `g` số nhóm trong thùng. Đúng nhưng không bảo đảm tuyến tính.

## 8. TRACE từng bước

Ta theo dõi thuật toán hai con trỏ trên `x = "bcab"`: đáp án xoay nhỏ nhất là `"abbc"` (bắt đầu từ chỉ số 2). Cột: `i`, `j`, `k`, hai ký tự so sánh và quyết định.

| cột | ý nghĩa |
|---|---|
| `i, j` | hai điểm bắt đầu ứng viên |
| `k` | số ký tự đã khớp liên tiếp |
| `a, b` | `x[(i+k)%n]`, `x[(j+k)%n]` |

In [12]:
def trace_min_rotation(x):
    n = len(x)
    print(f"x = {x!r}, n = {n}")
    i, j, k = 0, 1, 0
    step = 0
    while i < n and j < n and k < n:
        a, b = x[(i + k) % n], x[(j + k) % n]
        step += 1
        if a == b:
            print(f"#{step}: i={i} j={j} k={k}  a={a} b={b}  bằng -> k += 1")
            k += 1
            continue
        if a > b:
            print(f"#{step}: i={i} j={j} k={k}  a={a} b={b}  a > b -> loại i..i+k: i += {k + 1}")
            i += k + 1
        else:
            print(f"#{step}: i={i} j={j} k={k}  a={a} b={b}  a < b -> loại j..j+k: j += {k + 1}")
            j += k + 1
        if i == j:
            print("      i == j -> j += 1")
            j += 1
        k = 0
    p = min(i, j)
    print(f"dừng: i={i} j={j} k={k} -> bắt đầu tại {p}, xoay nhỏ nhất = {x[p:] + x[:p]!r}")
    return x[p:] + x[:p]

trace_min_rotation("bcab")
print()
trace_min_rotation("abab")

x = 'bcab', n = 4
#1: i=0 j=1 k=0  a=b b=c  a < b -> loại j..j+k: j += 1
#2: i=0 j=2 k=0  a=b b=a  a > b -> loại i..i+k: i += 1
#3: i=1 j=2 k=0  a=c b=a  a > b -> loại i..i+k: i += 1
      i == j -> j += 1
#4: i=2 j=3 k=0  a=a b=b  a < b -> loại j..j+k: j += 1
dừng: i=2 j=4 k=0 -> bắt đầu tại 2, xoay nhỏ nhất = 'abbc'

x = 'abab', n = 4
#1: i=0 j=1 k=0  a=a b=b  a < b -> loại j..j+k: j += 1
#2: i=0 j=2 k=0  a=a b=a  bằng -> k += 1
#3: i=0 j=2 k=1  a=b b=b  bằng -> k += 1
#4: i=0 j=2 k=2  a=a b=a  bằng -> k += 1
#5: i=0 j=2 k=3  a=b b=b  bằng -> k += 1
dừng: i=0 j=2 k=4 -> bắt đầu tại 0, xoay nhỏ nhất = 'abab'


'abab'

Ở `"abab"`: `k` tăng tới `n` (hai ứng viên cho cùng chuỗi, vì chuỗi tuần hoàn) rồi dừng; kết quả xoay nhỏ nhất là `"abab"` với `p = 0`. Giờ trace cả quá trình tính khoá cho từng từ của ví dụ 1:

In [13]:
def trace_key(w):
    E, O = w[::2], w[1::2]
    kE, kO = min_rotation_fast(E), min_rotation_fast(O)
    print(f"{w!r}: E={E!r} O={O!r}  -> khoá ({kE!r}, {kO!r})")
    return (kE, kO)

keys = [trace_key(w) for w in ["ntgwz", "zwntg"]]
print("số khoá khác nhau:", len(set(keys)))
for w in ["abc", "cab", "bac", "acb", "bca", "cba"]:
    trace_key(w)

'ntgwz': E='ngz' O='tw'  -> khoá ('gzn', 'tw')
'zwntg': E='zng' O='wt'  -> khoá ('gzn', 'tw')
số khoá khác nhau: 1
'abc': E='ac' O='b'  -> khoá ('ac', 'b')
'cab': E='cb' O='a'  -> khoá ('bc', 'a')
'bac': E='bc' O='a'  -> khoá ('bc', 'a')
'acb': E='ab' O='c'  -> khoá ('ab', 'c')
'bca': E='ba' O='c'  -> khoá ('ab', 'c')
'cba': E='ca' O='b'  -> khoá ('ac', 'b')


## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def minimumGroups(self, words: List[str]) -> int:
        def least_rotation(x: str) -> str:
            n = len(x)
            if n <= 1:
                return x
            i, j, k = 0, 1, 0
            while i < n and j < n and k < n:
                a, b = x[(i + k) % n], x[(j + k) % n]
                if a == b:
                    k += 1
                    continue
                if a > b:
                    i += k + 1
                else:
                    j += k + 1
                if i == j:
                    j += 1
                k = 0
            p = min(i, j)
            return x[p:] + x[:p]

        keys = set()
        for w in words:
            keys.add((least_rotation(w[::2]), least_rotation(w[1::2])))
        return len(keys)

for words, expected in EXAMPLES:
    assert Solution().minimumGroups(words) == expected
print("Code nộp khớp 3 ví dụ")

Code nộp khớp 3 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Phép biến đổi giữ nguyên hai thế giới.** Chỉ số chẵn chỉ nhận ký tự từ `E`, chỉ số lẻ từ `O`. `t = transform(s, a, b)` ⇔ `E_t = rot(E_s, a)` và `O_t = rot(O_s, b)` (có `a` và `b` độc lập).

**(b) Quan hệ tương đương.** Phản xạ (a = b = 0), đối xứng (`rot(·, −a)`), bắc cầu (`rot(rot(x, a), a') = rot(x, a + a')`). Vì vậy:

- một nhóm mà mọi cặp tương đương chỉ chứa các chuỗi **cùng lớp**;
- mỗi lớp đã là một nhóm hợp lệ.

Số nhóm tối thiểu = số lớp.

**(c) Dạng chuẩn tắc.** `rot(x, a) = y` với mọi `a` ⇔ `y` là một xoay vòng của `x` ⇔ `x` và `y` cùng tập `{x[i:] + x[:i]}` ⇔ `min` của hai tập bằng nhau. Vậy `lyndon(E_s) = lyndon(E_t)` ⇔ `E_t` là xoay vòng của `E_s` (và tương tự với `O`).

**(d) Thuật toán hai con trỏ đúng.** Khi so sánh xoay từ `i` và `j`, nếu `x[i+k] > x[j+k]` sau `k` ký tự khớp, thì với mọi `t ∈ [0, k]`, xoay từ `i + t` lớn hơn xoay từ `j + t` (hai xoay có tiền tố khớp `k − t` rồi ký tự `x[i+k] > x[j+k]`), nên `i, ..., i+k` đều không thể là điểm bắt đầu nhỏ nhất → bỏ qua cả đoạn. Đối xứng với `j`. Mỗi lần đẩy `i` hoặc `j` ít nhất `k + 1` còn mỗi lần khớp tăng `k` một, nên tổng số bước `≤ 2n + ...` tức `O(n)`. Khi `k` đạt `n`, hai xoay bằng nhau (chuỗi tuần hoàn) và `min(i, j)` là điểm bắt đầu hợp lệ.

**(e) Khoá phân biệt lớp.** Hai chuỗi khác độ dài có khoá khác nhau (độ dài `E`, `O` khác); cùng độ dài thì khoá là cặp dạng chuẩn. Hai chuỗi cùng khoá ⇔ cùng lớp, nên `len(set(keys))` là đáp án.

## 11. Phân tích Time/Space

Gọi `T` = tổng độ dài các chuỗi (`T ≤ 5·10^5`), `L` = độ dài một chuỗi.

| Cách | Time | Space |
|---|---|---|
| Vét cạn theo định nghĩa | `O(Bell(n) + n²·L³)` | `O(n²)` |
| Khoá min-rotation ngây thơ | `O(Σ L²)` | `O(T)` |
| Khoá hai con trỏ (nộp) | `O(T)` | `O(T)` |
| Thùng đa tập + nhân đôi | `O(T log T + Σ k·g·L)` | `O(T)` |

Đo thực tế với các dạng dữ liệu xấu nhất (tổng độ dài `5·10^5`):

In [15]:
import time, random

def timed(words):
    t0 = time.time()
    r = Solution().minimumGroups(words)
    return r, round(time.time() - t0, 2)

rng = random.Random(1)
T = 500000
print("1 chuỗi ngẫu nhiên dài 5e5          :", timed(["".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(T))]))
print("1 chuỗi toàn 'a' dài 5e5            :", timed(["a" * T]))
print("1 chuỗi 'ab' lặp dài 5e5            :", timed(["ab" * (T // 2)]))
print("1 chuỗi 'aaaa...ab' (xấu cho so sánh):", timed(["a" * (T - 1) + "b"]))
print("1e5 chuỗi độ dài 5 ngẫu nhiên       :", timed(["".join(rng.choice("abc") for _ in range(5)) for _ in range(100000)]))
print("50000 chuỗi độ dài 10 trên {a,b}     :", timed(["".join(rng.choice("ab") for _ in range(10)) for _ in range(50000)]))

1 chuỗi ngẫu nhiên dài 5e5          : (1, 0.1)


1 chuỗi toàn 'a' dài 5e5            : (1, 0.06)
1 chuỗi 'ab' lặp dài 5e5            : (1, 0.05)
1 chuỗi 'aaaa...ab' (xấu cho so sánh): (1, 0.06)


1e5 chuỗi độ dài 5 ngẫu nhiên       : (66, 0.08)
50000 chuỗi độ dài 10 trên {a,b}     : (64, 0.09)


## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ cho hai cách nghĩ sai, đối chiếu ngẫu nhiên giữa vét cạn ↔ khoá ngây thơ ↔ hai con trỏ ↔ thùng đa tập ↔ code nộp.

In [16]:
S = Solution()
for words, expected in EXAMPLES:
    for fn in (brute_groups, groups_naive, groups_fast, groups_bucket, S.minimumGroups):
        assert fn(words) == expected, (fn.__name__, words)
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [17]:
# Edge case
assert S.minimumGroups(["a"]) == 1
assert S.minimumGroups(["a", "a", "a"]) == 1            # trùng nhau
assert S.minimumGroups(["a", "b"]) == 2                 # độ dài 1, khác ký tự
assert S.minimumGroups(["ab", "ba"]) == 2               # độ dài 2: E, O chỉ có 1 ký tự -> không đổi
assert S.minimumGroups(["abc", "cba"]) == 1             # E = "ac" xoay
assert S.minimumGroups(["abcd", "cdab", "adcb"]) == 1   # abcd: (ac, bd); cdab: (ca, db); adcb: (ac, db) -> cùng lớp
assert S.minimumGroups(["aaaa", "aaaa"]) == 1
assert S.minimumGroups(["abab", "baba"]) == 2           # (aa, bb) khác (bb, aa)
assert S.minimumGroups(["abcde" * 2, "cdeab" * 2]) == 1   # E = acebd / cebda, O = bdace / daceb: đều là xoay của nhau
assert S.minimumGroups(["a" * 5, "a" * 4, "a" * 3]) == 3   # khác độ dài luôn khác nhóm
print("Edge case: OK")

Edge case: OK


**Phản ví dụ cho các cách nghĩ sai.**

1. **Chỉ so sánh đa tập** (sắp xếp `E` và `O`): bỏ qua việc xoay chỉ tạo ra `ℓ` hoán vị trong số `ℓ!`. Ví dụ `"axbxc"` (`E = "abc"`) và `"axcxb"` (`E = "acb"`) cùng đa tập nhưng **không** là xoay vòng của nhau.
2. **Xoay cả chuỗi** (`t in s + s`): ví dụ 1 của đề (`"ntgwz"`, `"zwntg"`) là cùng nhóm nhưng không phải xoay cả chuỗi của nhau.

In [18]:
def groups_multiset_only(words):
    return len({("".join(sorted(w[::2])), "".join(sorted(w[1::2]))) for w in words})

def groups_whole_rotation(words):
    reps = []
    for w in words:
        if not any(len(r) == len(w) and w in r + r for r in reps):
            reps.append(w)
    return len(reps)

w1 = ["axbxc", "axcxb"]
print("đa tập:", groups_multiset_only(w1), "| đúng:", S.minimumGroups(w1))        # 1 vs 2
w2 = EXAMPLES[0][0]
print("xoay cả chuỗi:", groups_whole_rotation(w2), "| đúng:", S.minimumGroups(w2))  # 2 vs 1

đa tập: 1 | đúng: 2
xoay cả chuỗi: 2 | đúng: 1


In [19]:
# 1) Đối chiếu ngẫu nhiên nhỏ với vét cạn theo định nghĩa (chia nhóm clique tối thiểu)
rng = random.Random(2026)
for it in range(220):
    n = rng.randint(1, 6)
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 7))) for _ in range(n)]
    ref = brute_groups(words)
    assert groups_naive(words) == ref, words
    assert groups_fast(words) == ref, words
    assert groups_bucket(words) == ref, words
    assert S.minimumGroups(words) == ref, words
print("220 input ngẫu nhiên nhỏ: vét cạn ↔ khoá ↔ thùng ↔ code nộp khớp")

220 input ngẫu nhiên nhỏ: vét cạn ↔ khoá ↔ thùng ↔ code nộp khớp


In [20]:
# 2) Kiểm chứng "tương đương" là quan hệ tương đương (bắc cầu, đối xứng) trên mọi bộ ba chuỗi ngắn
import itertools
pool = ["".join(p) for L in range(1, 6) for p in itertools.product("ab", repeat=L)]
sample = rng.sample(pool, 28)
for s_ in sample:
    assert equivalent(s_, s_)
for s_, t_ in itertools.permutations(sample, 2):
    assert equivalent(s_, t_) == equivalent(t_, s_)
for s_, t_, u_ in itertools.permutations(sample[:14], 3):
    if equivalent(s_, t_) and equivalent(t_, u_):
        assert equivalent(s_, u_)
print("phản xạ, đối xứng, bắc cầu: OK trên mẫu")

# 3) Đối chiếu khoá với định nghĩa: equivalent(s,t) <=> key(s) == key(t)
for s_, t_ in itertools.combinations(pool, 2):
    if len(s_) == len(t_):
        assert equivalent(s_, t_) == (key_naive(s_) == key_naive(t_)), (s_, t_)
print("equivalent(s, t) ⇔ cùng khoá: OK trên mọi cặp cùng độ dài ≤ 5 trên {a,b}")

phản xạ, đối xứng, bắc cầu: OK trên mẫu
equivalent(s, t) ⇔ cùng khoá: OK trên mọi cặp cùng độ dài ≤ 5 trên {a,b}


In [21]:
# 4) Kiểm tra min_rotation_fast với chuỗi ngẫu nhiên đa dạng (bảng chữ nhỏ, chuỗi tuần hoàn)
rng = random.Random(5)
for it in range(4000):
    L = rng.randint(1, 12)
    alpha = rng.choice(["a", "ab", "abc"])
    x = "".join(rng.choice(alpha) for _ in range(L))
    if rng.random() < 0.3:
        x = x[: rng.randint(1, len(x))] * rng.randint(1, 4)       # chuỗi tuần hoàn
    assert min_rotation_fast(x) == min_rotation_naive(x), x
print("4000 chuỗi: min_rotation_fast khớp ngây thơ")

# 5) Input vừa: các cách nhanh khớp nhau
rng = random.Random(77)
for it in range(25):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 40))) for _ in range(rng.randint(1, 60))]
    assert groups_naive(words) == groups_fast(words) == groups_bucket(words) == S.minimumGroups(words)
print("25 input vừa: khớp")

4000 chuỗi: min_rotation_fast khớp ngây thơ
25 input vừa: khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ so sánh đa tập ký tự (sắp xếp)

Gợi ý "sắp xếp `E` và `O`" nhìn hấp dẫn vì xoay vòng giữ nguyên đa tập. Nhưng đó chỉ là điều kiện **cần**.

In [22]:
words = ["axbxc", "axcxb"]          # E = "abc" và "acb": cùng đa tập, không là xoay của nhau
print("sai :", groups_multiset_only(words))      # 1
print("đúng:", Solution().minimumGroups(words))   # 2

sai : 1
đúng: 2


**Vì sao sai:** `"abc"` có đúng 3 xoay (`abc, bca, cab`), `"acb"` không nằm trong đó. **Cách đúng:** dùng dạng chuẩn tắc (xoay vòng nhỏ nhất) hoặc `y in x + x`.

### Lỗi 2 — Xoay cả chuỗi thay vì xoay hai dãy chẵn/lẻ độc lập

Đề cho phép xoay `E` và `O` với số lần **khác nhau**. Xoay cả chuỗi chỉ là trường hợp đặc biệt.

In [23]:
words = ["ntgwz", "zwntg"]          # ví dụ 1 của đề
print("sai :", groups_whole_rotation(words))     # 2
print("đúng:", Solution().minimumGroups(words))   # 1

sai : 2
đúng: 1


**Cách đúng:** tách `E = s[::2]`, `O = s[1::2]`, chuẩn hoá từng dãy.

### Lỗi 3 — Tính xoay vòng nhỏ nhất bằng `O(L²)`

`min(x[i:] + x[:i] for i in range(n))` đúng, và với chuỗi vài nghìn ký tự thì nhanh. Nhưng nó là `O(L²)`: thời gian tăng gấp 4 khi chuỗi dài gấp đôi, nên tới `L` cỡ trăm nghìn là bắt đầu tốn nhiều giây (đề cho tới `5·10^5`).

In [24]:
def groups_quadratic(words):
    return len({key_naive(w) for w in words})

x = "".join(random.Random(3).choice("ab") for _ in range(100000))
t0 = time.time(); r1 = min_rotation_naive(x); t_naive = time.time() - t0
t0 = time.time(); r2 = min_rotation_fast(x); t_fast = time.time() - t0
assert r1 == r2
print(f"độ dài 100000: ngây thơ O(L²) = {t_naive:.2f}s, hai con trỏ O(L) = {t_fast:.3f}s")
print("ước lượng cho L = 5e5 (tăng theo L²):", round(t_naive * (500000 / 100000) ** 2), "giây")

độ dài 100000: ngây thơ O(L²) = 0.66s, hai con trỏ O(L) = 0.015s
ước lượng cho L = 5e5 (tăng theo L²): 16 giây


**Cách đúng:** thuật toán hai con trỏ (hoặc Booth / dùng Duval) `O(L)`.

### Lỗi 4 — Quên xử lý chuỗi tuần hoàn / `i == j` trong hai con trỏ

Nếu không có `if i == j: j += 1`, hoặc không dừng khi `k == n`, vòng lặp có thể kẹt hoặc cho điểm bắt đầu sai trên chuỗi như `"abab"`, `"aaaa"`.

In [25]:
def buggy_min_rotation(x, limit=10000):
    n = len(x)
    i, j, k = 0, 1, 0
    steps = 0
    while i < n and j < n:                 # SAI: thiếu điều kiện k < n và thiếu 'if i == j'
        steps += 1
        if steps > limit:
            return None                    # vòng lặp vô hạn
        a, b = x[(i + k) % n], x[(j + k) % n]
        if a == b:
            k += 1
            continue
        if a > b: i += k + 1
        else: j += k + 1
        k = 0
    return x[min(i, j):] + x[:min(i, j)]

print("buggy 'abab':", buggy_min_rotation("abab"))          # None: kẹt vô hạn
print("đúng  'abab':", min_rotation_fast("abab"))
print("buggy 'abba':", buggy_min_rotation("abba"), "| đúng:", min_rotation_fast("abba"))

buggy 'abab': None
đúng  'abab': abab
buggy 'abba': None | đúng: aabb


**Cách đúng:** điều kiện lặp `i < n and j < n and k < n`, và `if i == j: j += 1` sau khi dịch con trỏ.

## 14. Mẹo & nhận dạng pattern

- **"Biến đổi bằng xoay vòng / hoán vị cố định"** → quan hệ tương đương theo nhóm; dùng **dạng chuẩn tắc** (canonical form) cho mỗi phần tử rồi đếm khoá khác nhau.
- **Khoá phải phân biệt hoàn toàn lớp.** Đa tập (sắp xếp) chỉ là **bất biến**, không phải bất biến **đầy đủ**; luôn tự hỏi "có hai thứ khác lớp mà cùng khoá không?".
- **Xoay vòng nhỏ nhất** = "necklace"/Lyndon rotation; thuật toán hai con trỏ `O(n)` rất ngắn, nên học thuộc.
- **Slicing `s[::2]`, `s[1::2]`** tách chẵn/lẻ nhanh và ngắn.
- **Tổng độ dài ≤ 5·10^5** nhưng một chuỗi có thể dài `5·10^5` → mọi thứ phải `O(L)` hoặc `O(L log L)`, tránh `O(L²)` và tránh nối chuỗi trong vòng lặp.
- **Nhóm "mọi cặp tương đương"** + quan hệ bắc cầu ⇒ số nhóm = số lớp; nếu quan hệ **không** bắc cầu thì phải cẩn thận (bài phủ clique, rất khó).

## 15. Biến thể

1. **Cho phép lật ngược** (xoay hoặc đảo chiều: "bracelet"): khoá = `min(lyndon(x), lyndon(x[::-1]))`.
2. **Chỉ xoay `E`**, `O` giữ nguyên: khoá = `(lyndon(E), O)`.
3. **Cho phép hoán vị tuỳ ý** (gợi ý đa tập): khoá = `(sorted(E), sorted(O))` — chính là điều mà ta đã thấy là **không đủ** cho bài gốc, nhưng đúng cho biến thể này.
4. **Số cặp tương đương** thay vì số nhóm: với mỗi lớp kích thước `c`, cộng `c(c−1)/2`.

Ta cài thử biến thể 1 (bracelet) và kiểm chứng bằng vét cạn định nghĩa.

In [26]:
def bracelet_key(x):
    return min(min_rotation_fast(x), min_rotation_fast(x[::-1]))

def groups_bracelet(words):
    return len({(bracelet_key(w[::2]), bracelet_key(w[1::2])) for w in words})

def equivalent_flip(s, t):
    if len(s) != len(t):
        return False
    E, O = split_eo(s)
    Et, Ot = split_eo(t)
    def ok(x, y):
        return y in {rotate_right(z, a) for z in (x, x[::-1]) for a in range(max(1, len(x)))}
    return ok(E, Et) and ok(O, Ot)

def brute_groups_flip(words):
    n = len(words)
    eq = [[equivalent_flip(words[i], words[j]) for j in range(n)] for i in range(n)]
    best = [n]; groups = []
    def dfs(i):
        if len(groups) >= best[0]: return
        if i == n:
            best[0] = len(groups); return
        for g in groups:
            if all(eq[i][j] for j in g):
                g.append(i); dfs(i + 1); g.pop()
        groups.append([i]); dfs(i + 1); groups.pop()
    dfs(0)
    return best[0]

rng = random.Random(15)
for _ in range(200):
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 9))) for _ in range(rng.randint(1, 6))]
    assert groups_bracelet(words) == brute_groups_flip(words), words
print("Biến thể bracelet: khớp vét cạn;", groups_bracelet(["abcde", "aedcb"]), "so với bài gốc", Solution().minimumGroups(["abcde", "aedcb"]))

Biến thể bracelet: khớp vét cạn; 2 so với bài gốc 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn định nghĩa | đồ thị tương đương + phủ clique | `O(Bell(n)·...)` | `O(n²)` | trọng tài, `n ≤ 7` |
| 2. Khoá min-rotation ngây thơ | `min(x[i:]+x[:i])` | `O(Σ L²)` | `O(T)` | chuỗi ngắn, trọng tài |
| 3. Hai con trỏ (nộp) | min-rotation tuyến tính | `O(T)` | `O(T)` | luôn dùng cho đề |
| 4. Thùng đa tập + `in x+x` | bộ lọc thô + kiểm tra | `O(T log T + ...)` | `O(T)` | minh hoạ bất biến cần-nhưng-chưa-đủ |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `"abcde"[::2]` và `"abcde"[1::2]` là gì? `rotate_right("abcd", 1)` là gì?

**Bài 2 (tính tay).** `words = ["abab", "baba", "abcd", "cdab"]`. Tính `(E, O)` của từng từ và số nhóm.

**Bài 3 (điền TODO).** Hoàn thiện `canon_pair(s)` trả về `(xoay vòng nhỏ nhất của E, xoay vòng nhỏ nhất của O)` dùng `min_rotation_fast`.

**Bài 4 (tự viết lại).** Viết `my_min_rotation(x)` bằng **hai con trỏ** `O(n)` (không nhìn code nộp), rồi `my_groups(words)`.

**Bài 5 (biến thể / mini challenge).** Đếm số **cặp** `(i, j)` với `i < j` mà `words[i]` và `words[j]` tương đương. Viết `count_equivalent_pairs(words)` bằng khoá lớp (không so sánh từng cặp) và kiểm chứng với vét cạn `equivalent`.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def canon_pair(s):
    # TODO 1: E = s[::2], O = s[1::2]
    # TODO 2: trả (min_rotation_fast(E), min_rotation_fast(O))
    pass

# Bài 4 — TODO
def my_min_rotation(x):
    # TODO: i, j, k = 0, 1, 0; vòng while i < n and j < n and k < n; so sánh x[(i+k)%n] với x[(j+k)%n]
    pass

def my_groups(words):
    # TODO: len(set(...)) của canon_pair
    pass

# Bài 5 — TODO
def count_equivalent_pairs(words):
    # TODO: đếm bằng Counter các khoá; cộng c * (c - 1) // 2 cho mỗi lớp
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chỉ số chẵn là 0, 2, 4; lẻ là 1, 3.
- Mức 2: `"abcde"[::2] = "ace"`, `"abcde"[1::2] = "bd"`.
- Mức 3: `rotate_right("abcd", 1) = "dabc"` (ký tự cuối lên đầu).

**Bài 2.**
- Mức 1: `abab`: `E = "aa"`, `O = "bb"`; `baba`: `E = "bb"`, `O = "aa"`.
- Mức 2: `abcd`: `(ac, bd)`; `cdab`: `(ca, db)`: xoay của nhau.
- Mức 3: ba lớp `{abab}`, `{baba}`, `{abcd, cdab}` → `3`.

**Bài 3.**
- Mức 1: tách bằng slicing.
- Mức 2: `min_rotation_fast` cho từng dãy.
- Mức 3: `return (min_rotation_fast(s[::2]), min_rotation_fast(s[1::2]))`.

**Bài 4.**
- Mức 1: ba biến `i, j, k`.
- Mức 2: khi `a > b` thì `i += k + 1`, ngược lại `j += k + 1`.
- Mức 3: nhớ `if i == j: j += 1` và đặt `k = 0`; kết quả `x[min(i,j):] + x[:min(i,j)]`.

**Bài 5.**
- Mức 1: khoá lớp như bài 3.
- Mức 2: `Counter(keys)`.
- Mức 3: `sum(c * (c - 1) // 2 for c in counter.values())`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert "abcde"[::2] == "ace" and "abcde"[1::2] == "bd"
assert rotate_right("abcd", 1) == "dabc"

# Bài 2
assert split_eo("abab") == ("aa", "bb") and split_eo("baba") == ("bb", "aa")
assert split_eo("abcd") == ("ac", "bd") and split_eo("cdab") == ("ca", "db")
assert Solution().minimumGroups(["abab", "baba", "abcd", "cdab"]) == 3

# Bài 3
def canon_pair_ans(s):
    return (min_rotation_fast(s[::2]), min_rotation_fast(s[1::2]))

assert canon_pair_ans("ntgwz") == canon_pair_ans("zwntg")

# Bài 4
def my_min_rotation_ans(x):
    n = len(x)
    if n <= 1:
        return x
    i, j, k = 0, 1, 0
    while i < n and j < n and k < n:
        a, b = x[(i + k) % n], x[(j + k) % n]
        if a == b:
            k += 1
            continue
        if a > b:
            i += k + 1
        else:
            j += k + 1
        if i == j:
            j += 1
        k = 0
    p = min(i, j)
    return x[p:] + x[:p]

def my_groups_ans(words):
    return len({(my_min_rotation_ans(w[::2]), my_min_rotation_ans(w[1::2])) for w in words})

rng = random.Random(33)
for _ in range(300):
    x = "".join(rng.choice("ab") for _ in range(rng.randint(0, 12)))
    assert my_min_rotation_ans(x) == min_rotation_naive(x)
for _ in range(100):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 8))) for _ in range(rng.randint(1, 10))]
    assert my_groups_ans(words) == Solution().minimumGroups(words)
print("Bài 1–4 OK")

Bài 1–4 OK


In [29]:
# Bài 5
from collections import Counter

def count_equivalent_pairs_ans(words):
    cnt = Counter((min_rotation_fast(w[::2]), min_rotation_fast(w[1::2])) for w in words)
    return sum(c * (c - 1) // 2 for c in cnt.values())

def count_equivalent_pairs_brute(words):
    return sum(1 for i in range(len(words)) for j in range(i + 1, len(words)) if equivalent(words[i], words[j]))

assert count_equivalent_pairs_ans(["abab", "baba", "abcd", "cdab"]) == 1
rng = random.Random(44)
for _ in range(150):
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 7))) for _ in range(rng.randint(1, 9))]
    assert count_equivalent_pairs_ans(words) == count_equivalent_pairs_brute(words), words
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm lớp bằng dạng chuẩn tắc xoay vòng":**

```python
def least_rotation(x):
    n = len(x)
    if n <= 1: return x
    i, j, k = 0, 1, 0
    while i < n and j < n and k < n:
        a, b = x[(i + k) % n], x[(j + k) % n]
        if a == b:
            k += 1; continue
        if a > b: i += k + 1
        else:     j += k + 1
        if i == j: j += 1
        k = 0
    p = min(i, j)
    return x[p:] + x[:p]

return len({(least_rotation(w[::2]), least_rotation(w[1::2])) for w in words})
```

**Checklist trước khi submit:**

- [ ] Tách chẵn/lẻ bằng `w[::2]` và `w[1::2]`; xử lý chuỗi độ dài 1 (dãy lẻ rỗng).
- [ ] Chuẩn hoá từng dãy bằng xoay vòng nhỏ nhất (không chỉ sắp xếp).
- [ ] Khoá là cặp `(chuẩn(E), chuẩn(O))`; đáp án là số khoá khác nhau.
- [ ] Thuật toán hai con trỏ có `k < n` và `if i == j: j += 1`.
- [ ] Tổng thời gian `O(Σ L)`; không dùng `O(L²)` cho chuỗi dài.
- [ ] Đã đối chiếu vét cạn trên input nhỏ và kiểm tra tính bắc cầu của quan hệ.